# 🚢 Ghost Fleet Detection — Mise en Jambe

**Hackathon Albert School 2026 — Sujet 4 : Détection d'activités maritimes anormales**

Ce notebook répond aux **12 questions** de la partie Mise en Jambe à partir des données AIS réduites.

---

| Partie | Questions | Thème |
|--------|-----------|-------|
| I | Q1 – Q4 | Exploration des données |
| II | Q5 – Q8 | Croisement et détection d'anomalies |
| III | Q9 – Q10 | Visualisation |
| IV | Q11 – Q12 | Analyse qualitative et recommandations |

**Fichiers utilisés :**
- `ais_data_small.csv` — positions AIS
- `suspicious_behaviors_small.csv` — comportements suspects
- `risk_zones_small.csv` — zones à risque

## ⚙️ Configuration & Imports

In [ ]:
import os
import math
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

# ── Localisation des données ────────────────────────────────────────────────
# Le notebook est dans V4/. Les données MiseEnJambe sont dans le dossier parent.
NOTEBOOK_DIR = Path.cwd()

_candidates = [
    NOTEBOOK_DIR / 'data-mise-en-jambe',
    NOTEBOOK_DIR.parent / 'corection_codex' / 'HackathonAlbert2026-main'
        / 'SujetsHackathon2026' / 'Sujet4' / 'MiseEnJambe',
]

DATA_DIR = None
for _c in _candidates:
    if _c.is_dir() and list(_c.glob('*.csv')):
        DATA_DIR = _c
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        'Dossier MiseEnJambe introuvable. '
        'Placez les CSV dans V4/data-mise-en-jambe/ ou vérifiez la structure du projet.'
    )

print(f'Données trouvées : {DATA_DIR}')
print('Fichiers CSV disponibles :', [f.name for f in DATA_DIR.glob('*.csv')])

## 📂 Chargement des données

In [ ]:
ais       = pd.read_csv(DATA_DIR / 'ais_data_small.csv', parse_dates=['timestamp'])
behaviors = pd.read_csv(DATA_DIR / 'suspicious_behaviors_small.csv')
zones     = pd.read_csv(DATA_DIR / 'risk_zones_small.csv')

print(f'AIS       : {ais.shape[0]:>6} lignes × {ais.shape[1]} colonnes')
print(f'Behaviors : {behaviors.shape[0]:>6} lignes × {behaviors.shape[1]} colonnes')
print(f'Zones     : {zones.shape[0]:>6} lignes × {zones.shape[1]} colonnes')

---
# Partie I — Exploration des données
## Q1 — Contenu et types de données de `ais_data_small.csv`

In [ ]:
print('=== Aperçu du DataFrame AIS ===')
display(ais)

In [ ]:
print('Types de colonnes :')
print(ais.dtypes)
print(f'\nDtype colonne timestamp : {ais["timestamp"].dtype}')
print(f'\nDimensions : {ais.shape[0]} lignes × {ais.shape[1]} colonnes')

### Interprétation Q1

Le fichier `ais_data_small.csv` contient des **messages AIS** (Automatic Identification System) émis par les navires :

| Colonne | Type | Description |
|---------|------|-------------|
| `mmsi` | str/int | Identifiant unique du navire (9 chiffres ou `FAKE-XXX`) |
| `timestamp` | datetime64 | Horodatage UTC de la position |
| `latitude` | float64 | Latitude WGS-84 (−90 à +90) |
| `longitude` | float64 | Longitude WGS-84 (−180 à +180) |
| `speed` | float64 | Vitesse sur le fond en nœuds (SOG) |
| `course` | float64 | Cap vrai en degrés (0–360) |
| `status` | str | Statut navigationnel AIS (Under Way, At Anchor, Moored…) |
| `ais_active` | bool | `True` = transpondeur actif, `False` = désactivé |

Le timestamp est correctement parsé en **datetime64[ns, UTC]**, ce qui permet des calculs temporels directs.

## Q2 — Lignes avec `ais_active = False` et MMSI uniques concernés

In [ ]:
ais_disabled = ais[ais['ais_active'].astype(str).str.lower() == 'false']
unique_mmsi_disabled = ais_disabled['mmsi'].unique()

print(f'Nombre de lignes où ais_active = False : {len(ais_disabled)}')
print(f'MMSI uniques concernés ({len(unique_mmsi_disabled)}) :')
for m in unique_mmsi_disabled:
    print(f'  → {m}')

display(ais_disabled[['mmsi', 'timestamp', 'latitude', 'longitude', 'speed', 'status', 'ais_active']])

### Interprétation Q2

Les lignes avec `ais_active = False` correspondent à des **messages reçus alors que le transpondeur était officiellement désactivé** (détectés par satellite ou station côtière malgré la désactivation volontaire).

**Pourquoi un navire désactiverait-il son AIS ?**
1. **Contournement de sanctions** — masquer une livraison vers un pays sous embargo
2. **Pêche illégale (IUU)** — opérer dans des zones protégées sans être repéré
3. **Transbordement illicite** — transfert de marchandises prohibées en mer
4. **Piraterie** — approcher une cible sans être détecté

> ⚠️ Chaque MMSI avec `ais_active = False` est un **signal d'alarme** à croiser avec les comportements suspects.

## Q3 — Comportements suspects : total, répartition par type, type le plus fréquent

In [ ]:
total_behaviors = len(behaviors)
type_counts     = behaviors['type'].value_counts()
most_frequent   = type_counts.idxmax()

print(f'Nombre total de comportements suspects : {total_behaviors}')
print(f'\nRépartition par type :')
for btype, cnt in type_counts.items():
    bar = '█' * cnt
    print(f'  {btype:<25} : {cnt:>3}  {bar}')
print(f'\nType le plus fréquent : {most_frequent} ({type_counts.max()} occurrence(s))')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#ef4444', '#f97316', '#eab308', '#22c55e', '#3b82f6', '#8b5cf6', '#ec4899']
type_counts.plot(kind='bar', ax=ax, color=colors[:len(type_counts)], edgecolor='white')
ax.set_xlabel('Type de comportement suspect', fontsize=12)
ax.set_ylabel('Nombre de cas', fontsize=12)
ax.set_title('Répartition des comportements suspects', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

## Q4 — Zones à risque : total et zones critiques

In [ ]:
total_zones    = len(zones)
critical_zones = zones[zones['risk_level'] == 'Critical']

print(f'Nombre total de zones : {total_zones}')
print(f'\nRépartition par niveau de risque :')
for level, cnt in zones['risk_level'].value_counts().items():
    print(f'  {level:<10} : {cnt}')

print(f'\nZones critiques ({len(critical_zones)}) :')
display(critical_zones[['zone_id', 'name', 'risk_level', 'description']])

### Interprétation Q4

Les zones à risque sont classées en 4 niveaux :

| Niveau | Signification |
|--------|---------------|
| **Critical** | Zone sous surveillance maximale (mer Rouge, golfe Persique, détroits) |
| **High** | Zone à risque élevé (routes de contrebande connues) |
| **Medium** | Zone de surveillance courante |
| **Low** | Zone de vigilance préventive |

Les **zones critiques** correspondent typiquement aux détroits stratégiques et aux eaux territoriales de pays sous sanctions.

---
# Partie II — Croisement et détection d'anomalies
## Q5 — Croisement suspects ↔ AIS (cast MMSI en string)

In [ ]:
ais_mmsi_set       = set(ais['mmsi'].astype(str))
behaviors_mmsi_arr = behaviors['mmsi'].astype(str).unique()
found_in_ais       = [m for m in behaviors_mmsi_arr if m in ais_mmsi_set]
absent_from_ais    = [m for m in behaviors_mmsi_arr if m not in ais_mmsi_set]

print(f'MMSI suspects uniques dans behaviors   : {len(behaviors_mmsi_arr)}')
print(f'Dont présents dans ais_data            : {len(found_in_ais)}')
print(f'Absents de ais_data (fantômes purs)    : {len(absent_from_ais)}')

print('\nDétail par MMSI suspect :')
for m in behaviors_mmsi_arr:
    status = '✅ présent' if m in ais_mmsi_set else '❌ absent '
    n_beh  = behaviors[behaviors['mmsi'].astype(str) == m].shape[0]
    print(f'  MMSI {m} → {status} dans AIS  ({n_beh} comportement(s))')

### Interprétation Q5

Le croisement entre les MMSI suspects et les données AIS révèle :
- Les navires **présents dans les deux sources** sont confirmés : leur comportement suspect peut être corrélé à leur trajectoire réelle.
- Les navires **absents du flux AIS** mais présents dans `behaviors` sont des **fantômes purs** : ils ont opéré en dehors de toute détection AIS.

> ⚠️ L'absence d'un navire suspect dans le flux AIS n'est **pas** un signal rassurant — c'est souvent le signe d'une désactivation délibérée prolongée.

## Q6 — Lignes AIS avec MMSI commençant par `FAKE-`

In [ ]:
fake_rows = ais[ais['mmsi'].astype(str).str.startswith('FAKE-')]

print(f'Nombre de lignes avec MMSI FAKE- : {len(fake_rows)}')

if not fake_rows.empty:
    print('\nDétail des lignes FAKE- :')
    display(fake_rows)
else:
    print('Aucune ligne FAKE- dans ce jeu de données.')
    print('(Les MMSI FAKE- sont des identifiants synthétiques injectés pour tester la détection du spoofing.)')

### Interprétation Q6 — Spoofing de MMSI

Un MMSI `FAKE-XXX` représente un **identifiant frauduleux** : le navire émet un MMSI qui n'appartient pas à un navire légalement enregistré.

**Risques du spoofing de MMSI :**
1. **Usurpation d'identité maritime** — se faire passer pour un navire légitime pour contourner les listes noires
2. **Danger pour la navigation** — les autres navires et les VTS (Vessel Traffic Services) voient une fausse identité
3. **Obstruction judiciaire** — traces falsifiées rendant impossible la reconstruction des mouvements
4. **Perturbation SAR** — MMSI inexistants bloquent les opérations de secours en mer

## Q7 — Détection des sauts de position (distance euclidienne > 0.05°)

In [ ]:
ais_sorted = ais.copy()
ais_sorted['mmsi_str']  = ais_sorted['mmsi'].astype(str)
ais_sorted['timestamp'] = pd.to_datetime(ais_sorted['timestamp'], utc=True, errors='coerce')
ais_sorted = ais_sorted.sort_values(['mmsi_str', 'timestamp'])

jump_records = []
for mmsi_val, group in ais_sorted.groupby('mmsi_str'):
    group = group.reset_index(drop=True)
    for i in range(1, len(group)):
        lat1 = group.loc[i - 1, 'latitude']
        lon1 = group.loc[i - 1, 'longitude']
        lat2 = group.loc[i,     'latitude']
        lon2 = group.loc[i,     'longitude']
        dist = math.sqrt((lat2 - lat1) ** 2 + (lon2 - lon1) ** 2)
        if dist > 0.05:
            jump_records.append({
                'mmsi'        : mmsi_val,
                'timestamp'   : group.loc[i, 'timestamp'],
                'lat_from'    : lat1,
                'lon_from'    : lon1,
                'lat_to'      : lat2,
                'lon_to'      : lon2,
                'distance_deg': round(dist, 4),
            })

df_jumps = pd.DataFrame(jump_records)
print(f'Nombre total de sauts > 0.05° : {len(df_jumps)}')

if not df_jumps.empty:
    display(df_jumps)
else:
    print('Aucun saut de position détecté avec ce seuil.')

### Interprétation Q7

Un **saut de position** (`distance > 0.05°` soit ≈ 5,5 km entre deux points consécutifs) est anormal car :
- Les messages AIS sont émis toutes les **2 à 10 secondes** à pleine vitesse
- Un déplacement de 5,5 km en quelques secondes impliquerait une vitesse de plusieurs **milliers de nœuds** → physiquement impossible

**Causes possibles :**
- **Injection de fausses coordonnées GPS** (GPS spoofing)
- **Perte de signal temporaire** avec reprise à un autre endroit
- **Erreur de transmission** du transpondeur

> La **distance de Haversine** (sur sphère) est plus précise que la distance euclidienne, mais pour des distances < 100 km, l'approximation euclidienne est acceptable.

## Q8 — Traversées de zones critiques (bounding box)

In [ ]:
def parse_bbox(coord_str):
    """Retourne (lat_min, lon_min, lat_max, lon_max) depuis 'lat1,lon1;lat2,lon2'."""
    parts      = str(coord_str).replace(' ', '').split(';')
    lat1, lon1 = map(float, parts[0].split(','))
    lat2, lon2 = map(float, parts[1].split(','))
    return min(lat1, lat2), min(lon1, lon2), max(lat1, lat2), max(lon1, lon2)


crossings = []
for _, zone_row in critical_zones.iterrows():
    lat_min, lon_min, lat_max, lon_max = parse_bbox(zone_row['coordinates'])
    for _, ais_row in ais.iterrows():
        lat = ais_row['latitude']
        lon = ais_row['longitude']
        if lat_min <= lat <= lat_max and lon_min <= lon <= lon_max:
            crossings.append({
                'mmsi'      : ais_row['mmsi'],
                'timestamp' : ais_row['timestamp'],
                'zone'      : zone_row['name'],
                'risk_level': zone_row['risk_level'],
                'lat'       : lat,
                'lon'       : lon,
            })

df_crossings = pd.DataFrame(crossings)
print(f'Nombre total de traversées de zones critiques : {len(df_crossings)}')

if not df_crossings.empty:
    display(df_crossings)
else:
    print('Aucune position AIS ne tombe dans une zone critique.')
    print('(Les positions AIS sont dans l\'Atlantique Nord-Est ;')
    print(' les zones critiques sont typiquement en mer Rouge et dans le golfe Persique.)')

### Interprétation Q8

La méthode **bounding box** est une approximation rapide :
- **Avantage** : O(n × z) complexité, simple à implémenter
- **Limite** : une bounding box rectangulaire peut inclure des zones hors de la région réelle

Une zone critique traversée par un navire suspect constitue un **signal fort** : croisement géographique + comportemental.

**Amélioration possible** : utiliser des polygones GeoJSON précis (shapely + geopandas) pour une détection exacte.

---
# Partie III — Visualisation
## Q9 — Statistiques des vitesses + histogramme

In [ ]:
speed        = ais['speed'].dropna()
speed_mean   = speed.mean()
speed_median = speed.median()
speed_std    = speed.std()
speed_max    = speed.max()

print('=== Statistiques des vitesses (nœuds) ===')
print(f'Moyenne    : {speed_mean:.2f}')
print(f'Mediane    : {speed_median:.2f}')
print(f'Ecart-type : {speed_std:.2f}')
print(f'Maximum    : {speed_max:.2f}')
print(f'Positions > 25 nœuds (anomalie) : {(speed > 25).sum()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Histogramme
axes[0].hist(speed, bins=15, color='steelblue', edgecolor='white', alpha=0.85, label='Fréquence')
axes[0].axvline(speed_mean,   color='red',    linestyle='--', linewidth=2,
                label=f'Moyenne : {speed_mean:.1f} nd')
axes[0].axvline(speed_median, color='green',  linestyle='--', linewidth=2,
                label=f'Médiane : {speed_median:.1f} nd')
axes[0].axvline(25,           color='black',  linestyle=':',  linewidth=2,
                label='Seuil anomalie : 25 nd')
axes[0].set_xlabel('Vitesse (nœuds)', fontsize=12)
axes[0].set_ylabel('Nombre de positions', fontsize=12)
axes[0].set_title('Distribution des vitesses AIS', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(axis='y', alpha=0.4)

# Boxplot par statut
statuses = ais['status'].dropna().unique()
data_by_status = [ais[ais['status'] == s]['speed'].dropna().values for s in statuses]
axes[1].boxplot(data_by_status, labels=statuses, patch_artist=True,
                boxprops=dict(facecolor='lightblue', color='steelblue'))
axes[1].axhline(25, color='red', linestyle='--', linewidth=1.5, label='Seuil 25 nd')
axes[1].set_xlabel('Statut navigational', fontsize=12)
axes[1].set_ylabel('Vitesse (nœuds)', fontsize=12)
axes[1].set_title('Vitesse par statut AIS', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(fontsize=10)
axes[1].grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('histogramme_vitesses.png', dpi=150)
plt.show()
print('Histogramme sauvegardé -> histogramme_vitesses.png')

### Interprétation Q9

Le seuil d'anomalie de **25 nœuds** est calibré sur les vitesses maximales des navires commerciaux :
- Cargo standard : 12–18 nœuds
- Tanker : 10–16 nœuds
- Pétrolier VLCC : 8–14 nœuds
- Porte-conteneurs rapide : 20–24 nœuds

Une vitesse > 25 nœuds pour un cargo est **physiquement improbable** et indique soit une **injection de données frauduleuses**, soit un **dysfonctionnement du transpondeur**.

## Q10 — Carte Folium interactive de la flotte fantôme

In [ ]:
suspicious_mmsi_set = set(behaviors['mmsi'].astype(str))
fake_mmsi_set       = set(ais[ais['mmsi'].astype(str).str.startswith('FAKE-')]['mmsi'].astype(str))

center_lat = ais['latitude'].mean()
center_lon = ais['longitude'].mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=6,
    tiles='CartoDB positron',
)

fg_normal     = folium.FeatureGroup(name='Positions normales',    show=True)
fg_disabled   = folium.FeatureGroup(name='AIS désactivé',         show=True)
fg_fake       = folium.FeatureGroup(name='MMSI FAKE (spoofing)',  show=True)
fg_suspicious = folium.FeatureGroup(name='Navires suspects',      show=True)
fg_zones      = folium.FeatureGroup(name='Zones à risque',        show=True)

ZONE_COLORS = {'Critical': 'red', 'High': 'orange', 'Medium': 'yellow', 'Low': 'green'}

# Zones de risque en rectangles
for _, zrow in zones.iterrows():
    lat_min, lon_min, lat_max, lon_max = parse_bbox(zrow['coordinates'])
    color = ZONE_COLORS.get(zrow['risk_level'], 'gray')
    folium.Rectangle(
        bounds=[[lat_min, lon_min], [lat_max, lon_max]],
        color=color, fill=True, fill_color=color, fill_opacity=0.15, weight=2,
        popup=folium.Popup(
            f"<b>{zrow['name']}</b><br>Risque : <b>{zrow['risk_level']}</b><br>{zrow['description']}",
            max_width=300),
        tooltip=f"{zrow['name']} ({zrow['risk_level']})",
    ).add_to(fg_zones)

# Positions AIS
for _, row in ais.iterrows():
    mmsi_str   = str(row['mmsi'])
    lat, lon   = row['latitude'], row['longitude']
    popup_html = (
        f"<b>MMSI :</b> {mmsi_str}<br>"
        f"<b>Timestamp :</b> {row['timestamp']}<br>"
        f"<b>Vitesse :</b> {row['speed']} nœuds<br>"
        f"<b>Cap :</b> {row['course']}°<br>"
        f"<b>Statut :</b> {row['status']}<br>"
        f"<b>AIS actif :</b> {row['ais_active']}"
    )

    if mmsi_str.startswith('FAKE-'):
        folium.Marker(
            location=[lat, lon],
            icon=folium.Icon(color='red', icon='exclamation-sign', prefix='glyphicon'),
            popup=folium.Popup(f"<b>ALERTE SPOOFING</b><br>{popup_html}", max_width=300),
            tooltip=f"FAKE MMSI: {mmsi_str}",
        ).add_to(fg_fake)

    elif str(row['ais_active']).lower() == 'false':
        folium.CircleMarker(
            location=[lat, lon], radius=7,
            color='orange', fill=True, fill_color='orange', fill_opacity=0.85,
            popup=folium.Popup(f"<b>AIS DÉSACTIVÉ</b><br>{popup_html}", max_width=300),
            tooltip=f"AIS OFF: {mmsi_str}",
        ).add_to(fg_disabled)

    elif mmsi_str in suspicious_mmsi_set:
        folium.CircleMarker(
            location=[lat, lon], radius=7,
            color='darkred', fill=True, fill_color='darkred', fill_opacity=0.85,
            popup=folium.Popup(f"<b>NAVIRE SUSPECT</b><br>{popup_html}", max_width=300),
            tooltip=f"Suspect: {mmsi_str}",
        ).add_to(fg_suspicious)

    else:
        folium.CircleMarker(
            location=[lat, lon], radius=5,
            color='gray', fill=True, fill_color='gray', fill_opacity=0.5,
            popup=folium.Popup(popup_html, max_width=300),
            tooltip=mmsi_str,
        ).add_to(fg_normal)

for fg in [fg_zones, fg_normal, fg_disabled, fg_suspicious, fg_fake]:
    fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

legend_html = """
<div style="position:fixed;bottom:40px;left:40px;z-index:1000;
     background:white;padding:12px 16px;border-radius:8px;
     border:2px solid #aaa;font-size:13px;
     box-shadow:3px 3px 6px rgba(0,0,0,0.2);">
  <b>Légende</b><br>
  <span style="color:gray;">&#9679;</span> Position normale<br>
  <span style="color:orange;">&#9679;</span> AIS désactivé<br>
  <span style="color:red;">&#9679;</span> MMSI FAKE (spoofing)<br>
  <span style="color:darkred;">&#9679;</span> Navire suspect<br>
  <hr style="margin:4px 0;">
  <span style="color:red;">&#9644;</span> Zone Critique<br>
  <span style="color:orange;">&#9644;</span> Zone Haute<br>
  <span style="color:goldenrod;">&#9644;</span> Zone Moyenne<br>
  <span style="color:green;">&#9644;</span> Zone Faible
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

map_path = 'carte_flotte_fantome.html'
m.save(map_path)
print(f'Carte sauvegardée -> {map_path}')
m

---
# Partie IV — Analyse qualitative et recommandations
## Q11 — Analyse comportementale : AIS désactivé et spoofing MMSI

### Pourquoi un navire désactiverait-il son AIS ? (≥ 4 raisons)

---

**1. Contournement de sanctions internationales**
Livrer des cargaisons vers des pays sous embargo (pétrole iranien, armes nord-coréennes, etc.) sans apparaître dans les registres AIS publics. Source : rapport OFAC 2022 sur les tankers fantômes.

**2. Pêche illégale (IUU — Illegal, Unreported, Unregulated fishing)**
Masquer sa présence dans des zones de pêche protégées ou des eaux territoriales étrangères pour échapper aux patrouilles des gardes-côtes.

**3. Transbordements illicites en mer (Ship-to-Ship)**
Transférer des marchandises prohibées (drogues, pétrole sanctionné) d'un navire à l'autre sans laisser de trace dans les registres AIS. Les deux navires impliqués désactivent simultanément leur transpondeur.

**4. Piraterie et activités criminelles**
Des navires pirates coupent leur AIS pour approcher des cibles sans être détectés. Des victimes peuvent aussi l'éteindre par panique.

**5. Opérations de renseignement clandestines**
Des navires étatiques ou para-étatiques désactivent leur AIS pour mener des opérations de surveillance, de sabotage (câbles sous-marins) ou de collecte de renseignement.

---

### Risques du spoofing de MMSI (≥ 3 risques)

**1. Usurpation d'identité maritime**
Un navire sanctionné peut se faire passer pour un navire légitime, contournant les contrôles douaniers, portuaires et les listes noires OFAC/ONU.

**2. Dangers pour la sécurité de la navigation**
De fausses positions induisent en erreur les autres navires et les systèmes de contrôle du trafic (VTS), augmentant le risque de collisions dans les détroits et chenaux étroits.

**3. Entrave aux enquêtes judiciaires**
Les traces de position falsifiées rendent impossible la reconstruction fidèle des mouvements d'un navire, bloquant les enquêtes sur le trafic, la piraterie ou la pollution maritime.

**4. Perturbation des opérations SAR (Search & Rescue)**
Des MMSI dupliqués ou inexistants compliquent l'identification des navires en détresse et ralentissent les secours en mer.

## Q12 — Champs manquants et algorithme de détection des sauts de position

### Champs manquants pour améliorer la détection (≥ 4)

---

**1. Numéro IMO**
Identifiant permanent du navire, indépendant du MMSI, permettant de détecter les changements frauduleux de MMSI ou de pavillon. Un navire peut changer de MMSI en 24h, mais son IMO reste fixe à vie.

**2. Destination et ETA déclarés**
Comparer la route réellement suivie à la destination officielle pour repérer les déviations vers des zones sous sanctions ou non déclarées.

**3. Identité du propriétaire / opérateur (bénéficiaire effectif)**
Relier les navires à des entités sanctionnées même après changement de pavillon, de nom commercial ou de société écran.

**4. Données satellitaires indépendantes (SAT-AIS / imagerie radar SAR)**
Vérifier que le navire est physiquement là où il prétend être et détecter les fausses positions envoyées par le transpondeur AIS.

**5. Historique des escales portuaires**
Croiser les ports visités avec les listes de ports à risque ou sous embargo pour identifier les comportements atypiques.

**6. Tirant d'eau (draught)**
Un écart entre le tirant déclaré et celui attendu selon la route peut signaler un transbordement illicite en mer (cargaison non déclarée).

### Algorithme de détection automatique des sauts de position

---

```
ENTRÉE : flux de messages AIS (temps réel ou batch)

ÉTAPE 1 — Collecte et nettoyage
  ├── Filtrer les messages malformés : MMSI non numérique, coordonnées
  │   hors limites (|lat| > 90 ou |lon| > 180), vitesse négative
  └── Normaliser les timestamps en UTC

ÉTAPE 2 — Tri chronologique par MMSI
  ├── GROUP BY mmsi
  └── ORDER BY timestamp ASC (dans chaque groupe)

ÉTAPE 3 — Calcul des déplacements entre points consécutifs
  Pour chaque paire (P_i, P_{i+1}) du même navire :
    distance    = haversine(lat1, lon1, lat2, lon2)   [km]
    delta_t     = t_{i+1} - t_i                       [secondes]
    v_implicite = distance / delta_t × 1.944           [nœuds]

ÉTAPE 4 — Règles de détection des anomalies
  ├── Saut de position      : distance > seuil_distance (ex. 5 km en 1 h)
  ├── Vitesse impossible    : v_implicite > 30 nœuds pour un cargo
  ├── Téléportation         : grande distance avec delta_t très court
  └── Trou de signal        : delta_t > seuil_silence (ex. 2 heures)

ÉTAPE 5 — Scoring et alertes
  ├── Attribuer un score de risque cumulatif par navire
  ├── Pondérer selon le type d'anomalie et le niveau de confiance
  ├── Déclencher une alerte si score > seuil_alerte
  └── Croiser avec les listes de navires sanctionnés et les zones à risque

ÉTAPE 6 — Visualisation et rapport
  ├── Afficher les trajectoires suspectes sur une carte interactive
  ├── Générer un rapport automatique (HTML/PDF) pour les analystes
  └── Alimenter un tableau de bord en temps réel pour le suivi continu

SORTIE : liste de navires suspects avec score de risque + alertes géolocalisées
```

### Implémentation Python de l'algorithme (Haversine)

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine_km(lat1, lon1, lat2, lon2):
    """Distance en km entre deux points GPS via la formule de Haversine."""
    R = 6371  # rayon moyen de la Terre en km
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))


def detect_position_jumps(ais_df, max_speed_knots=30, min_gap_hours=2):
    """
    Détecte les sauts de position anormaux dans le flux AIS.

    Paramètres
    ----------
    ais_df          : DataFrame avec colonnes mmsi, timestamp, latitude, longitude
    max_speed_knots : vitesse maximale physiquement possible (défaut : 30 nœuds)
    min_gap_hours   : durée minimale pour un 'trou de signal' (défaut : 2 h)

    Retourne
    --------
    DataFrame des anomalies détectées
    """
    df = ais_df.copy()
    df['mmsi']      = df['mmsi'].astype(str)
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')
    df = df.sort_values(['mmsi', 'timestamp'])

    anomalies = []
    for mmsi_val, group in df.groupby('mmsi'):
        group = group.reset_index(drop=True)
        for i in range(1, len(group)):
            r0, r1 = group.iloc[i-1], group.iloc[i]
            dist_km   = haversine_km(r0.latitude, r0.longitude, r1.latitude, r1.longitude)
            delta_sec = (r1.timestamp - r0.timestamp).total_seconds()
            if delta_sec <= 0:
                continue
            v_knots = dist_km / delta_sec * 1944  # km/s → nœuds
            gap_h   = delta_sec / 3600

            if v_knots > max_speed_knots:
                anomalies.append({'mmsi': mmsi_val, 'type': 'Speed Jump',
                                   'v_knots': round(v_knots, 1), 'dist_km': round(dist_km, 2),
                                   'timestamp': r1.timestamp})
            if gap_h > min_gap_hours:
                anomalies.append({'mmsi': mmsi_val, 'type': 'Signal Gap',
                                   'gap_hours': round(gap_h, 2), 'dist_km': round(dist_km, 2),
                                   'timestamp': r1.timestamp})

    return pd.DataFrame(anomalies)


# Application sur les données réduites
df_algo_anomalies = detect_position_jumps(ais)
print(f'Anomalies détectées par l\'algorithme Haversine : {len(df_algo_anomalies)}')
if not df_algo_anomalies.empty:
    print(df_algo_anomalies.groupby('type').size().to_string())
    display(df_algo_anomalies.head(10))

---
## Récapitulatif final

In [ ]:
print('=' * 60)
print('  RÉCAPITULATIF — Métriques clés (Mise en Jambe)')
print('=' * 60)
print(f"""
  Lignes AIS totales                   : {len(ais)}
  Lignes avec AIS désactivé            : {len(ais_disabled)}
  MMSI uniques avec AIS désactivé      : {len(unique_mmsi_disabled)}
  Lignes avec MMSI FAKE-               : {len(fake_rows)}

  Comportements suspects totaux        : {total_behaviors}
  Type le plus fréquent                : {most_frequent} ({type_counts.max()} occurrence(s))
  Navires suspects trouvés dans AIS    : {len(found_in_ais)} / {len(behaviors_mmsi_arr)}

  Zones à risque totales               : {total_zones}
  Zones critiques                      : {len(critical_zones)}

  Sauts de position (> 0.05°)          : {len(df_jumps)}
  Traversées de zones critiques        : {len(df_crossings)}

  Vitesse moyenne                      : {speed_mean:.2f} nœuds
  Vitesse médiane                      : {speed_median:.2f} nœuds
  Écart-type vitesses                  : {speed_std:.2f} nœuds

  Fichiers générés :
    -> histogramme_vitesses.png
    -> carte_flotte_fantome.html
""")